In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inicialização de Qubits

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Quando um circuito é executado em uma unidade de processamento quântico (QPU) da IBM&reg;, um reset implícito é tipicamente inserido no início do circuito para garantir que os qubits sejam inicializados em zero. Isso é controlado pela flag `init_qubits`, definida como uma [opção de execução de primitiva](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

No entanto, o processo de reset não é perfeito, levando a erros de preparação de estado. Para atenuar o erro, o sistema também insere um tempo de atraso de repetição (ou `rep_delay`) entre os circuitos. Cada backend tem um `rep_delay` padrão diferente, mas geralmente é maior que T1 para permitir que o ambiente resete os qubits. O `rep_delay` padrão pode ser consultado executando `backend.default_rep_delay`.

Todas as QPUs da IBM usam execução com taxa de repetição dinâmica, o que permite que você altere o `rep_delay` para cada job. Os circuitos que você envia em um job de primitiva são agrupados para execução na QPU. Esses circuitos são executados iterando sobre os circuitos para cada shot solicitado; a execução é feita por coluna sobre uma matriz de circuitos e shots, conforme ilustrado na figura a seguir.

![A primeira coluna representa o shot 0. Os circuitos são executados em ordem, de 0 a 3. A segunda coluna representa o shot 1. Os circuitos são executados em ordem, de 0 a 3. As colunas restantes seguem o mesmo padrão. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matriz de execução por coluna")

Como o `rep_delay` é inserido entre os circuitos, cada shot da execução encontra esse atraso. Portanto, diminuir o `rep_delay` reduz o tempo total de execução na QPU, mas à custa de uma maior taxa de erro de preparação de estado, como pode ser visto na imagem a seguir:

![Esta imagem mostra que, à medida que o valor de `rep_delay` é reduzido, a taxa de erro de preparação de estado aumenta.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Atraso de repetição versus taxa de erro")

Definir tanto `rep_delay=0` quanto `init_qubits=False` essencialmente "mescla" os circuitos, já que os qubits começarão no estado final do shot anterior.

Observe que, embora os circuitos em um job de primitiva sejam agrupados para execução na QPU, não há garantia sobre a ordem em que os circuitos dos PUBs são executados. Assim, mesmo que você envie `pubs=[pub1, pub2]`, não há garantia de que os circuitos de `pub1` serão executados antes dos de `pub2`. Também não há garantia de que circuitos do mesmo job serão executados como um único lote na QPU.

## Especificar rep_delay para um job de primitiva

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Próximos passos
> **Tip:** - Experimente um exemplo no tutorial [Algoritmo de otimização aproximada quântica (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Veja como [começar a usar primitivas.](./get-started-with-primitives)